## Preparation

In [1]:
from utils import setup_dataset_and_graphs
data_dir = "data/FB15k-237+H"
dataset, graph_train, graph_valid, graph_test = setup_dataset_and_graphs(data_dir, logging=True, add_reverse=False)

Loaded 14505 nodes from data/FB15k-237+H/id2ent.pkl.
Loaded 474 relations from data/FB15k-237+H/id2rel.pkl.
Loaded 14951 node titles from data/FB15k-237+H/extra/entity2text.txt.
Loaded 544230 edges from data/FB15k-237+H/train.txt, skipped 0 edges due to missing nodes or relations.
Loaded 579282 edges from data/FB15k-237+H/valid.txt, skipped 0 edges due to missing nodes or relations.
Loaded 620158 edges from data/FB15k-237+H/test.txt, skipped 0 edges due to missing nodes or relations.


In [2]:
from utils import load_all_queries
query_dataset, query_dataset_hard = load_all_queries(dataset, data_dir, "test", version=2)

Loading queries for the complete dataset ...
Loaded queries from data/FB15k-237+H/test-queries.pkl
Loaded hard answers from data/FB15k-237+H/test-hard-answers.pkl
Loaded easy answers from data/FB15k-237+H/test-easy-answers.pkl
Loading queries for the hard dataset ...
Loaded queries from data/FB15k-237+H/test-queries.pkl
Loaded hard answers from data/FB15k-237+H/test-hard-answers.pkl
Loaded easy answers from data/FB15k-237+H/test-easy-answers.pkl
Finished loading queries.


In [3]:
k = 10
t_norm, t_conorm = "prod", "prod"
model_path = "models/FB15k-237-model-rank-1000-epoch-100-1602508358.pt"

In [4]:
from symbolic_torch import SymbolicReasoning
from xcqa_torch import XCQA

reasoner = SymbolicReasoning(graph_valid, logging=False)
xcqa = XCQA(symbolic=reasoner, dataset=dataset, logging=False, model_path=model_path, normalize=False)

ComplEx(
  (embeddings): ModuleList(
    (0): Embedding(14505, 2000, sparse=True)
    (1): Embedding(474, 2000, sparse=True)
  )
)


## Query Sampling

In [5]:
from utils import get_num_atoms

query_type = "2p"
num_atoms = get_num_atoms(query_type)
queries_hard = query_dataset_hard.get_queries(query_type)
queries_complete = query_dataset.get_queries(query_type)

In [6]:
query_idx = 136
query_hard = queries_hard[query_idx]
query_complete = queries_complete[query_idx]
query_hard

Query(type=2p, query=((1697, (414, 64)),), answer=[6151, 2830, 7190, 2973, 290, 182, 7480, 8888, 5438, 1602, 4427, 5708, 3276, 3034, 93, 1504, 618, 4346, 6508, 4726, 12410, 4092])

In [7]:
from utils import human_readable

hr = human_readable(query_hard, dataset, fol=False, full_relation=True)
fol = human_readable(query_hard, dataset, fol=True, full_relation=True)
print(hr)
print(fol)

Time Warner (1697) --[/organization/organization/child./organization/organization_relationship/child (414)]--> V0
V0 --[/film/film_distributor/films_distributed./film/film_film_distributor_relationship/film (64)]--> V1
?V2: ∃V1 · /organization/organization/child./organization/organization_relationship/child(Time Warner, V0) ∧ /film/film_distributor/films_distributed./film/film_film_distributor_relationship/film(V0, V1)


In [11]:
hard_answers = query_hard.get_answer()
complete_answers = query_complete.get_answer()
easy_answers = list(set(complete_answers) - set(hard_answers))
print(f"There are {len(hard_answers)} hard answers, {len(easy_answers)} easy answers, resulting in {len(complete_answers)} complete answers.")

There are 22 hard answers, 292 easy answers, resulting in 314 complete answers.


In [9]:
for hard_answer in query_hard.get_answer():
    print(f"{hard_answer}: {dataset.get_title_by_id(hard_answer)}")

6151: Terminator 3: Rise of the Machines
2830: The Impossible
7190: The Contender
2973: J. Edgar
290: One Flew Over the Cuckoo's Nest
182: Babylon 5: The Gathering
7480: Willy Wonka & the Chocolate Factory
8888: Superman/Batman: Public Enemies
5438: Tootsie
1602: The Animatrix
4427: The Others
5708: Wag the Dog
3276: A Few Good Men
3034: Match Point
93: The Passion of the Christ
1504: Sherlock Holmes
618: Striptease
4346: Sherlock Holmes: A Game of Shadows
6508: 2046
4726: Iron Man
12410: Edge of Darkness
4092: The Queen


In [12]:
from shapley import Shapley
shapley = Shapley(xcqa)

## CQD-SHAP Explanations

### Target Answer: The Animatrix

In [13]:
from utils import compute_rank

target = 1602
result_cqd = xcqa.query_execution(query_hard, k=k, coalition=[1] * num_atoms, t_norm=t_norm, t_conorm=t_conorm)
result_sym = xcqa.query_execution(query_hard, k=k, coalition=[0] * num_atoms, t_norm=t_norm, t_conorm=t_conorm)
rank_cqd = compute_rank(result_cqd, complete_answers, target)
rank_sym = compute_rank(result_sym, complete_answers, target)
print(f"Rank of {target} ({dataset.get_title_by_id(target)}) in CQD: {rank_cqd}")
print(f"Rank of {target} ({dataset.get_title_by_id(target)}) in Symbolic: {rank_sym}")

Rank of 1602 (The Animatrix) in CQD: 14
Rank of 1602 (The Animatrix) in Symbolic: 9811


In [14]:
filtered_nodes = list(set(complete_answers) - set([target]))
shapleys = shapley.shapley_values(query_hard, filtered_nodes, target)
shapleys

{0: -166.0, 1: 9963.0}

#### Analysis

In [15]:
anchor = query_hard.atoms[0]['head']
relation1 = query_hard.atoms[0]['relation']
relation2 = query_hard.atoms[1]['relation']
print(f"Anchor: {dataset.get_title_by_id(anchor)}")
print(f"Relation 1: {dataset.get_relation_by_id(relation1)}")
print(f"Relation 2: {dataset.get_relation_by_id(relation2)}")

Anchor: Time Warner
Relation 1: +/organization/organization/child./organization/organization_relationship/child
Relation 2: +/film/film_distributor/films_distributed./film/film_film_distributor_relationship/film


In [16]:
print("Tail entities for anchor - relation 1 -> ? in the valid graph:")
count = 0
for edge in graph_valid.edges:
    if edge.get_head().get_id() == anchor and edge.get_id() == relation1:
        count += 1
        print(edge.get_tail().get_title())
print(count)

Tail entities for anchor - relation 1 -> ? in the valid graph:
HBO
Castle Rock Entertainment
Atari Games
Warner Bros. Interactive Entertainment
HBO Films
The CW
Warner Music Group
Warner Bros. Entertainment
AOL
Lorimar Television
New Line Cinema
Village Roadshow Pictures
Adult Swim
EMI
14


In [17]:
print("Tail entities for anchor - relation 1 -> ? in the test graph:")
count = 0
for edge in graph_test.edges:
    if edge.get_head().get_id() == anchor and edge.get_id() == relation1:
        count += 1
        print(edge.get_tail().get_title())
print(count)

Tail entities for anchor - relation 1 -> ? in the test graph:
HBO
Castle Rock Entertainment
Atari Games
Warner Bros. Interactive Entertainment
HBO Films
The CW
Warner Music Group
Warner Bros. Entertainment
AOL
Lorimar Television
New Line Cinema
Village Roadshow Pictures
Adult Swim
EMI
Warner Home Video
15


In [18]:
print("Head entities for ? - relation 2 -> target in the valid graph:")
count = 0
for edge in graph_valid.edges:
    if edge.get_tail().get_id() == target and edge.get_id() == relation2:
        count += 1
        print(edge.get_head().get_title())
print(count)

Head entities for ? - relation 2 -> target in the valid graph:
Warner Home Video
1


In [19]:
print("Head entities for ? - relation 2 -> target in the test graph:")
count = 0
for edge in graph_test.edges:
    if edge.get_tail().get_id() == target and edge.get_id() == relation2:
        count += 1
        print(edge.get_head().get_title())
print(count)

Head entities for ? - relation 2 -> target in the test graph:
Warner Home Video
1


In [21]:
# Perform link prediction for the first atom (anchor - relation 1 -> ?)
lp_results = xcqa.link_prediction.predict(anchor, relation1, return_df=True)
lp_results['tail_id'] = lp_results.index
lp_results["tail_title"] = lp_results["tail_id"].apply(lambda x: dataset.get_title_by_id(x))
lp_results

,score,tail_id,tail_title
3127,9.539072,3127,HBO
13652,9.324352,13652,Atari Games
12567,9.296451,12567,AOL
3679,9.277428,3679,Lorimar Television
13519,9.257105,13519,Warner Bros. Interactive Entertainment
...,...,...,...
3817,-1.605688,3817,Marianne Faithfull
2859,-1.616714,2859,Tony Award for Best Musical
5468,-1.619688,5468,Andrew Jackson
1943,-1.641539,1943,Jorge Luis Borges


In [22]:
print("The top-10 tail entities predicted by the link prediction model for anchor - relation 1 -> ? are:")
lp_results['tail_title'][:10].tolist()

The top-10 tail entities predicted by the link prediction model for anchor - relation 1 -> ? are:


['HBO',
 'Atari Games',
 'AOL',
 'Lorimar Television',
 'Warner Bros. Interactive Entertainment',
 'Village Roadshow Pictures',
 'The CW',
 'New Line Cinema',
 'HBO Films',
 'Warner Music Group']